In [9]:
import torch
import torch.nn as nn
import torch.nn.init as init

# ──────────────────────────────────────────────
# 1. Weight initialization functions
# ──────────────────────────────────────────────

def init_xavier_uniform(model: nn.Module) -> nn.Module:
    """Xavier/Glorot uniform — best for Sigmoid & Tanh activations."""
    for name, param in model.named_parameters():
        if "weight" in name and param.dim() >= 2:
            init.xavier_uniform_(param)
        elif "bias" in name:
            init.zeros_(param)
    return model


def init_xavier_normal(model: nn.Module) -> nn.Module:
    """Xavier/Glorot normal — same use-case, normal distribution variant."""
    for name, param in model.named_parameters():
        if "weight" in name and param.dim() >= 2:
            init.xavier_normal_(param)
        elif "bias" in name:
            init.zeros_(param)
    return model


def init_he_uniform(model: nn.Module) -> nn.Module:
    """He/Kaiming uniform — best for ReLU & LeakyReLU activations."""
    for name, param in model.named_parameters():
        if "weight" in name and param.dim() >= 2:
            init.kaiming_uniform_(param, nonlinearity="relu")
        elif "bias" in name:
            init.zeros_(param)
    return model


def init_he_normal(model: nn.Module) -> nn.Module:
    """He/Kaiming normal — same use-case, normal distribution variant."""
    for name, param in model.named_parameters():
        if "weight" in name and param.dim() >= 2:
            init.kaiming_normal_(param, nonlinearity="relu")
        elif "bias" in name:
            init.zeros_(param)
    return model


# ──────────────────────────────────────────────
# 2. Cleaner approach: apply per-layer with a hook
# ──────────────────────────────────────────────

def smart_init(module: nn.Module):
    """
    Apply the RIGHT initializer based on the activation that follows.
    Use with model.apply(smart_init).
    """
    if isinstance(module, nn.Linear):
        # Default to He (safe for ReLU-based nets)
        init.kaiming_normal_(module.weight, nonlinearity="relu")
        if module.bias is not None:
            init.zeros_(module.bias)

    elif isinstance(module, nn.Conv2d):
        init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
        if module.bias is not None:
            init.zeros_(module.bias)

    elif isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d)):
        init.ones_(module.weight)       # scale  = 1
        init.zeros_(module.bias)        # offset = 0


# ──────────────────────────────────────────────
# 3. Helper: compare stats before & after init
# ──────────────────────────────────────────────

def print_weight_stats(model: nn.Module, label: str = ""):
    """Print mean / std of every weight tensor."""
    print(f"── {label} ──" if label else "── Weight stats ──")
    for name, p in model.named_parameters():
        if "weight" in name:
            print(f"  {name:<22}  mean={p.data.mean():+.5f}  std={p.data.std():.5f}")
    print()


# ──────────────────────────────────────────────
# 4. Example usage
# ──────────────────────────────────────────────

if __name__ == "__main__":

    # --- A simple network to experiment with ---
    class DemoNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.Sequential(
                nn.Linear(64, 128),
                nn.ReLU(),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, 1),
                nn.Sigmoid(),
            )

        def forward(self, x):
            return self.layers(x)

    # ── Compare different initializations ──

    # A) PyTorch default
    torch.manual_seed(0)
    model_default = DemoNet()
    print_weight_stats(model_default, "PyTorch Default (Kaiming uniform)")

    # B) Xavier uniform
    torch.manual_seed(0)
    model_xavier = DemoNet()
    init_xavier_uniform(model_xavier)
    print_weight_stats(model_xavier, "Xavier Uniform")

    # C) He normal
    torch.manual_seed(0)
    model_he = DemoNet()
    init_he_normal(model_he)
    print_weight_stats(model_he, "He Normal")

    # D) smart_init via .apply()
    torch.manual_seed(0)
    model_smart = DemoNet()
    model_smart.apply(smart_init)
    print_weight_stats(model_smart, "smart_init (.apply)")

    # ── Show how init affects forward pass signal ──
    print("── Forward pass activation magnitudes ──")
    x = torch.randn(32, 64)  # batch of 32

    for label, model in [("Default", model_default),
                         ("Xavier",  model_xavier),
                         ("He",      model_he)]:
        h = x
        for i, layer in enumerate(model.layers):
            h = layer(h)
            if isinstance(layer, nn.ReLU):
                print(f"  {label:<8} after layer {i}: "
                      f"mean={h.mean().item():+.4f}  std={h.std().item():.4f}")
        print()

    # ── Quick rule of thumb ──
    print("Quick guide:")
    print("  • Sigmoid / Tanh  →  Xavier (xavier_uniform_ or xavier_normal_)")
    print("  • ReLU / variants →  He     (kaiming_uniform_ or kaiming_normal_)")
    print("  • BatchNorm       →  weight=1, bias=0")
    print("  • Bias            →  zeros")

── PyTorch Default (Kaiming uniform) ──
  layers.0.weight         mean=-0.00025  std=0.07191
  layers.2.weight         mean=+0.00082  std=0.05132
  layers.4.weight         mean=+0.00077  std=0.07147
  layers.6.weight         mean=+0.00134  std=0.10769

── Xavier Uniform ──
  layers.0.weight         mean=+0.00107  std=0.10147
  layers.2.weight         mean=+0.00037  std=0.10179
  layers.4.weight         mean=-0.00229  std=0.14451
  layers.6.weight         mean=-0.02601  std=0.26604

── He Normal ──
  layers.0.weight         mean=-0.00297  std=0.17503
  layers.2.weight         mean=-0.00088  std=0.12500
  layers.4.weight         mean=-0.00273  std=0.17484
  layers.6.weight         mean=+0.03836  std=0.30940

── smart_init (.apply) ──
  layers.0.weight         mean=-0.00297  std=0.17503
  layers.2.weight         mean=-0.00088  std=0.12500
  layers.4.weight         mean=-0.00273  std=0.17484
  layers.6.weight         mean=+0.03836  std=0.30940

── Forward pass activation magnitudes ──
  De